In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head(3)

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title(f"Target Distribution: Delivery_Time")
plt.xlabel("Delivery_Time")
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns='Order_ID')

df.head(1) # For checking only

In [ ]:
# Task 2: Write your code here:
# df.isnull().sum() # This is the closer look :)

df_clean = df.copy()
null_col = ['Weather', 'Traffic_Level', 'Courier_Experience_yrs', 'Time_of_Day']
for col in null_col:
    df_clean[col] = df_clean[col].fillna('unknown')

# (df_clean.isnull().sum().sum())

df_clean = df_clean.dropna()

df_clean.isnull().sum()


In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_clean.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")
df_clean.shape

In [ ]:
x1=[]
x2=[]
x3=[]
x4=[]
for i in df_clean['Weather']:
  x1.append(i)
for i in df_clean['Traffic_Level']:
  x2.append(i)
for i in df_clean['Time_of_Day']:
  x3.append(i)
for i in df_clean['Vehicle_Type']:
  x4.append(i)
x11= []
x22= []
x33= []
x44 = []
for i in x1:
  if x11.count(i) < 1:
    x11.append(i)
for i in x2:
  if x22.count(i) < 1:
    x22.append(i)
for i in x3:
  if x33.count(i) < 1:
    x33.append(i)
for i in x4:
  if x44.count(i) < 1:
    x44.append(i)
print(x11,x22,x33,x44)



In [ ]:
from numpy import dtype
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder
df_clean['Weather'] = df_clean['Weather'].map({'Windy':1, 'Clear':2, 'Foggy':3, 'Rainy':4, 'Snowy':5, 'unknown':6})
df_clean['Traffic_Level'] = df_clean['Traffic_Level'].map({'Low':1, 'Medium':2, 'High':3, 'unknown':6})
df_clean['Time_of_Day'] = df_clean['Time_of_Day'].map({'Afternoon':1, 'Evening':2, 'Night':3, 'Morning':4, 'unknown':6})
df_clean['Vehicle_Type'] = df_clean['Vehicle_Type'].map({'Scooter':1, 'Bike':2, 'Car':3,'unknown':6})

df_clean.head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")
scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()


In [ ]:
# Task 6: Write your code here:
import numpy as np

plt.hist(np.sqrt(df_clean["Delivery_Time"]),bins=30)
plt.show()

df_clean["Delivery_Time"] = np.sqrt(df_clean["Delivery_Time"])

In [ ]:
X = df.drop("Delivery_Time", axis=1) # FEATURES to meajure the target column
y = df['Delivery_Time'] # THE TARGET COLUMN

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

In [ ]:


def gradient_descent(X, y, learning_rate, n_iters=500): # LERANING RATE = ALPHA
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m # X.T means x transpose
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
# float_col = ['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Preparation_Time_min','Delivery_Time']
# for i in float_col:
#   df_clean[i] = df_clean[i].dtype()

df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].map({'unknown':0})

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

feature_cols = [ 'Distance_km' ,'Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min', 'Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)

# kfold = KFold(n_splits=5, shuffle=True, random_state=42)


# model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
# model.fit(X_train_scaled, y_train)

# X_train, X_test, y_train, y_test = kfold(X, y, test_size=0.2, random_state=42)

# mae_scores = []
# rmse_scores = []

# for train_idx, val_idx in kfold.split(X_train_scaled):
#     X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
#     y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

#     # Train and predict
#     model.fit(X_fold_train, y_fold_train)
#     y_fold_pred = model.predict(X_fold_val)

#     # Calculate metrics
#     mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
#     rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

# mae_scores = np.array(mae_scores)
# rmse_scores = np.array(rmse_scores)

# print(f"5-Fold CV Results:")
# print(f"MAE:  ${mae_scores.mean():,.2f}")
# print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 2,3,4,5: Write your code here:

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: